# Prediction-powered corpus estimation: how to use it

`ppi_corpus.py` implements the estimator of the paper (Algorithm 1). It returns the rate of a property in a corpus, such as the fraction of case reports that describe Long COVID, with a 95% confidence interval. It needs:

* an LLM yes-probability for **every** document, and
* expert labels for a **small random sample** of documents (the gold set).

This demo uses `data/bench_llm/hoc.csv`: 5,345 sentences of the Hallmarks of Cancer benchmark scored by the paper's zero-shot LLM (Qwen3.8-27B). Every sentence also has an expert label, so we can hide most labels, run the estimator, and check the answer against the true rate.

In [1]:
import numpy as np
import pandas as pd

import ppi_corpus as pc

pd.set_option("display.precision", 4)

## 1. Load the data

In [2]:
df = pd.read_csv("data/bench_llm/hoc.csv")      # one row per sentence
print(df.shape)
df.head()

(5345, 3)


,label,pred,p_yes
0,0,0,9.1105e-04
1,0,0,1.2219e-05
2,1,0,9.0280e-05
3,1,0,1.0322e-03
4,0,0,2.6119e-04


| column | meaning |
|---|---|
| `label` | expert label (1 = the sentence mentions a hallmark of cancer) |
| `pred` | the LLM's hard yes/no answer |
| `p_yes` | the LLM's yes-probability, $p_{\mathrm{raw}}(x)$ in Eq. (11) |

## 2. Input format

Draw a random gold set of $n = 100$ sentences and keep their expert labels. The other $N$ sentences form the unlabeled pool, and their labels are hidden from the estimator.

| input | shape | meaning |
|---|---|---|
| `y_gold` | `(n,)` 0/1 | expert labels of the gold set |
| `p_gold` | `(n,)` float in [0, 1] | LLM yes-probability of each gold document |
| `p_pool` | `(N,)` float in [0, 1] | LLM yes-probability of each unlabeled document |
| `folds` *(optional)* | `(n,)` int | report-level fold of each gold document for cross-fitting (default: random, `K=5`) |
| `calibration` *(optional)* | `(p_cal, y_cal)` | a labeled set **independent** of the gold set, used only to fit Platt scaling |

For your own corpus this is one gold file (`id, label, p_yes`) and one pool file (`id, p_yes`).
The gold set must be a random sample of the same corpus as the pool, because this is the condition under which the interval covers (Sec. IV-B).

In [3]:
rng = np.random.default_rng(0)
truth = df.label.mean()                         # known here only because this is a benchmark

gold = rng.choice(len(df), 100, replace=False)
pool = np.setdiff1d(np.arange(len(df)), gold)

y_gold = df.label.values[gold]
p_gold, p_pool = df.p_yes.values[gold], df.p_yes.values[pool]
print(f"n = {len(y_gold)} gold, N = {len(p_pool)} pool, true rate = {truth:.4f}")

n = 100 gold, N = 5245 pool, true rate = 0.2541


## 3. Run Algorithm 1

With no independent calibration set, the gold set is split into $K = 5$ folds. Platt scaling is fit out-of-fold, so no gold label calibrates its own pseudo-label, and the estimate is Cross-PPI (Eqs. 15, 16, 20).

In [4]:
res = pc.estimate_corpus_rate(y_gold, p_gold, p_pool, K=5, seed=0)
print(res["method"])
print(f"estimate {res['estimate']:.4f}   95% CI [{res['ci'][0]:.4f}, {res['ci'][1]:.4f}]   score shift {res['score_shift']:+.4f}")

cross-fitted Platt (K=5) + Cross-PPI
estimate 0.3281   95% CI [0.2467, 0.4094]   score shift +0.0181


## 4. Output format

| key | meaning |
|---|---|
| `estimate` | estimated corpus rate $\hat\theta$ |
| `ci` | `(lower, upper)` 95% interval, intersected with [0, 1] as in Eq. (21) |
| `lam` | power-tuning weight $\lambda$ (estimated by PPI++; 1 for Cross-PPI) |
| `score_shift` | $\hat\Delta_f$ of Eq. (26): mean pool pseudo-label minus mean gold pseudo-label. A large value flags a gold set that does not look like the pool |
| `method` | which branch of Algorithm 1 ran |
| `f_gold`, `f_pool` | the pseudo-labels the estimator used |

## 5. The other branches

* **Independent calibration set** (`calibration=(p_cal, y_cal)`): Platt is fit once, then PPI++ with an estimated $\lambda$.
* **Raw score, no calibration** (`ppi_plusplus`): PPI does not need calibrated scores. This is the estimator in the LLM blocks of Table II.
* **Hard pseudo-labels** (`hard_label_ppi`): $h(x) = 1\{p_{\mathrm{raw}}(x) \ge t_0\}$, with $t_0$ fixed in advance.

In [5]:
cal = rng.choice(pool, 200, replace=False)                 # 200 extra labeled sentences, used only for Platt
rest = np.setdiff1d(pool, cal)
p_rest = df.p_yes.values[rest]

rows = {
    "cross-fitted Platt + Cross-PPI": res,
    "independent Platt + PPI++": pc.estimate_corpus_rate(y_gold, p_gold, p_rest,
                                                         calibration=(df.p_yes.values[cal], df.label.values[cal])),
    "raw score + PPI++": pc.ppi_plusplus(y_gold, p_gold, p_pool),
    "hard labels + PPI++": pc.hard_label_ppi(y_gold, p_gold, p_pool, t0=0.5),
}

## 6. Baselines (Sec. V-C)

* **Naive count** of the LLM's yes answers on the pool. Its interval is narrow but centred on the LLM's own rate, which is biased.
* **Gold-only**: the 100 expert labels alone.
* **ACC** (Adjusted Classify-and-Count): corrects the count with the LLM's TPR and FPR measured on the gold set, using a bootstrap interval.

In [6]:
h_gold, h_pool = df.pred.values[gold], df.pred.values[pool]
rows["naive count"] = pc.naive_ci(h_pool)
rows["gold-only"] = pc.gold_only_ci(y_gold)
lo, hi = pc.acc_ci(y_gold, h_gold, h_pool, rng=0)
rows["ACC"] = dict(estimate=pc.acc_estimate(y_gold, h_gold, h_pool.mean()), ci=(lo, hi))

pd.DataFrame({name: dict(estimate=r["estimate"], lower=r["ci"][0], upper=r["ci"][1],
                         width=r["ci"][1] - r["ci"][0], covers_truth=r["ci"][0] <= truth <= r["ci"][1])
              for name, r in rows.items()}).T

,estimate,lower,upper,width,covers_truth
cross-fitted Platt + Cross-PPI,0.3281,0.2467,0.4094,0.1628,True
independent Platt + PPI++,0.3238,0.2425,0.4051,0.1626,True
raw score + PPI++,0.3229,0.2444,0.4013,0.1569,True
hard labels + PPI++,0.3104,0.2292,0.3917,0.1626,True
naive count,0.1825,0.172,0.1929,0.0209,False
gold-only,0.31,0.2194,0.4006,0.1813,True
ACC,0.3445,0.1809,0.6264,0.4455,True


This is a single random draw. This gold sample happens to contain 31% positives against a true rate of 25%, so every estimate that uses the gold labels lands high; the PPI intervals still contain the truth, while the naive interval is centred on the LLM's biased rate and misses it. Coverage is the fraction of many such draws whose interval contains the truth; `reproduce.ipynb` measures it over 500 draws per gold-set size (Table II).

## 7. Candidate screen (Algorithm 1, lines 1–2)

The screen only decides which documents are reviewed first, and the estimator always runs on the full pool. Here are the Long COVID screen scores: similarity to a Long COVID prototype and a regex-cue flag, for the 123 gold reports and the 3,066-report pool.

In [7]:
sg, sp = pd.read_csv("data/screen_gold.csv"), pd.read_csv("data/screen_pool.csv")
tau = pc.choose_threshold(sg.embed_sim, sg.label, recall=1.0)     # keep every gold positive
keep = pc.union_screen(sp.embed_sim, sp.regex_hit, tau)
print(f"tau = {tau:.2f}; the union screen keeps {keep.sum():,} of {len(sp):,} pool reports")

tau = 0.40; the union screen keeps 2,383 of 3,066 pool reports
